# ZotVision — CNN-ViT Hybrid Classifier
**EfficientNet-B4 + Google ViT** · 4-class firefighter hazard detection  
Classes: `null` · `hazard` · `person` · `both`

Runs a hyperparameter sweep, auto-selects the best config, and saves everything to a unique Google Drive folder.

## 1 · Install dependencies

In [ ]:
!pip install -q efficientnet_pytorch transformers kagglehub seaborn scikit-learn

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/ZotVision'

import os
os.makedirs(GDRIVE_ROOT, exist_ok=True)
print(f'Google Drive root: {GDRIVE_ROOT}')

## 3 · Kaggle credentials

In [ ]:
# Option A — upload kaggle.json interactively
# from google.colab import files
# files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B — set credentials inline (DO NOT commit real tokens)
import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'
print('Kaggle credentials ready.')

## 4 · Imports & config

In [ ]:
import hashlib, json, os, random, shutil, time

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from efficientnet_pytorch import EfficientNet
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTModel, ViTConfig

# ── Paths ──
KAGGLE_DATASET = 'varshvijay05/firefighter-hazard-dataset'
DATASET_DIR    = '/content/dataset'
IMAGES_DIR     = os.path.join(DATASET_DIR, 'images')
LABELS_FILE    = os.path.join(DATASET_DIR, 'results', 'labels.txt')
RESULTS_DIR    = os.path.join(DATASET_DIR, 'results')

# ── Labels ──
NUM_CLASSES = 4
LABEL_MAP   = {'null': 0, 'hazard': 1, 'person': 2, 'both': 3}
ID_TO_LABEL = {0: 'null', 1: 'hazard', 2: 'person', 3: 'both'}
CLASS_NAMES = [ID_TO_LABEL[i] for i in range(NUM_CLASSES)]

IMG_SIZE = 224
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

try:
    GDRIVE_ROOT
except NameError:
    GDRIVE_ROOT = '/content/drive/MyDrive/ZotVision'

print(f'Device: {DEVICE}')

## 5 · Dataset

In [ ]:
def ensure_dataset():
    if os.path.isdir(IMAGES_DIR) and os.path.isfile(LABELS_FILE):
        print('[INFO] Dataset already present.')
        return
    print(f'[INFO] Downloading {KAGGLE_DATASET} from Kaggle...')
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_DATASET)
    for src, dst in [(os.path.join(path, 'images'), IMAGES_DIR),
                     (os.path.join(path, 'results'), os.path.dirname(LABELS_FILE))]:
        if os.path.isdir(src):
            os.makedirs(dst, exist_ok=True)
            for f in os.listdir(src):
                shutil.copy2(os.path.join(src, f), dst)
    print(f'[INFO] Dataset ready at {DATASET_DIR}')

ensure_dataset()

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples, self.transform = samples, transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label


def load_samples(labels_file=LABELS_FILE, images_dir=IMAGES_DIR):
    samples = []
    with open(labels_file) as f:
        for i, line in enumerate(f):
            name = line.strip()
            if not name or name not in LABEL_MAP: continue
            p = os.path.join(images_dir, f'{i+1}.jpg')
            if os.path.exists(p): samples.append((p, LABEL_MAP[name]))
    return samples


def get_transforms(train):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.2),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


all_samples = load_samples()
random.seed(42)
random.shuffle(all_samples)
split = int(0.8 * len(all_samples))
train_samples, val_samples = all_samples[:split], all_samples[split:]
print(f'Total: {len(all_samples)}  |  Train: {len(train_samples)}  |  Val: {len(val_samples)}')

## 6 · Model — CNN-ViT Hybrid

In [ ]:
class CNNViTHybrid(nn.Module):
    """
    EfficientNet-B4 feature extractor → patch projection → CLS token
    → positional embedding → Google ViT encoder → MLP classifier
    """
    def __init__(self, efficientnet_variant='efficientnet-b4', vit_hidden_size=768,
                 vit_num_layers=6, vit_num_heads=12, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.cnn = EfficientNet.from_pretrained(efficientnet_variant)
        cnn_ch = self.cnn._conv_head.out_channels
        self.cnn._avg_pooling = self.cnn._dropout = self.cnn._fc = nn.Identity()
        for name, p in self.cnn.named_parameters():
            if not any(k in name for k in ['_blocks.28','_blocks.29','_blocks.30','_blocks.31','_conv_head']):
                p.requires_grad = False

        self.patch_proj = nn.Conv2d(cnn_ch, vit_hidden_size, kernel_size=1)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, vit_hidden_size))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_embed  = nn.Parameter(torch.zeros(1, 50, vit_hidden_size))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.vit_hidden_size = vit_hidden_size

        self.vit_encoder = ViTModel(ViTConfig(
            hidden_size=vit_hidden_size, num_hidden_layers=vit_num_layers,
            num_attention_heads=vit_num_heads, intermediate_size=vit_hidden_size*4,
            hidden_dropout_prob=dropout, attention_probs_dropout_prob=dropout,
            image_size=IMG_SIZE, patch_size=16, num_channels=3,
        ))
        self.classifier = nn.Sequential(
            nn.LayerNorm(vit_hidden_size), nn.Linear(vit_hidden_size, 256),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(256, num_classes),
        )

    def forward(self, x):
        B    = x.size(0)
        feat = self.patch_proj(self.cnn.extract_features(x))
        N    = feat.shape[2] * feat.shape[3]
        feat = feat.flatten(2).transpose(1, 2)
        tokens = torch.cat([self.cls_token.expand(B,-1,-1), feat], dim=1)
        if self.pos_embed.size(1) != N+1:
            pos = nn.functional.interpolate(
                self.pos_embed.transpose(1,2), size=N+1, mode='linear', align_corners=False
            ).transpose(1,2)
        else:
            pos = self.pos_embed
        tokens = tokens + pos
        out = self.vit_encoder.encoder(tokens)
        cls = self.vit_encoder.layernorm(out.last_hidden_state)[:, 0, :]
        return self.classifier(cls)

## 7 · Training utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate_per_class(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        all_preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

print('Training utilities defined.')

## 8 · Per-label accuracy & confusion heatmap

In [ ]:
def compute_per_class_accuracy(preds, labels, class_names=CLASS_NAMES):
    report_dict = classification_report(labels, preds, target_names=class_names, digits=4, output_dict=True)
    print(classification_report(labels, preds, target_names=class_names, digits=4))
    per_class_acc = {cls: round(report_dict[cls]['recall'], 4) for cls in class_names}
    df = pd.DataFrame([
        {'Label': cls, 'Accuracy': per_class_acc[cls],
         'Precision': round(report_dict[cls]['precision'], 4),
         'F1-score':  round(report_dict[cls]['f1-score'],  4),
         'Support':   int(report_dict[cls]['support'])}
        for cls in class_names
    ]).set_index('Label')
    display(df.style.background_gradient(cmap='YlGn', subset=['Accuracy', 'F1-score']))
    return per_class_acc


def plot_confusion_heatmap(preds, labels, class_names=CLASS_NAMES, save_path=None):
    cm      = confusion_matrix(labels, preds, labels=range(len(class_names)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, fmt, cmap, title in [
        (axes[0], cm,      'd',   'Blues',   'Confusion Matrix (counts)'),
        (axes[1], cm_norm, '.2f', 'Oranges', 'Confusion Matrix (normalized)'),
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'Heatmap saved → {save_path}')
    plt.show()

print('Metrics helpers defined.')

## 9 · Google Drive helpers
Each `save_to_gdrive()` call creates `MyDrive/ZotVision/iter<N>_<hash>/` — never overwrites a previous run.

In [ ]:
def _next_iter_folder(gdrive_root=GDRIVE_ROOT):
    index_path = os.path.join(gdrive_root, 'iterations.json')
    index = json.load(open(index_path)) if os.path.exists(index_path) else {'count': 0, 'runs': []}
    index['count'] += 1
    n = index['count']
    h = hashlib.sha1(f'{n}-{time.time()}'.encode()).hexdigest()[:6]
    folder = f'iter{n}_{h}'
    index['runs'].append({'iter': n, 'hash': h, 'folder': folder})
    os.makedirs(gdrive_root, exist_ok=True)
    json.dump(index, open(index_path, 'w'), indent=2)
    return os.path.join(gdrive_root, folder)


def save_to_gdrive(results_dir, gdrive_root=GDRIVE_ROOT):
    iter_dir = _next_iter_folder(gdrive_root)
    os.makedirs(iter_dir, exist_ok=True)
    print(f'\n[GDRIVE] Saving to: {iter_dir}')
    for fname in ['model_weights.pth', 'hyperparam_results.json', 'run_comparison.png', 'best_config.json']:
        src = os.path.join(results_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(iter_dir, fname))
            print(f'  ✓ {fname}')
    for fname in sorted(os.listdir(results_dir)):
        if fname.startswith(('model_run', 'heatmap_run')):
            shutil.copy2(os.path.join(results_dir, fname), os.path.join(iter_dir, fname))
            print(f'  ✓ {fname}')
    print('[GDRIVE] Done.')
    return iter_dir

print('Google Drive helpers defined.')

## 10 · Hyperparameter grid
8 configs covering a range of learning rates, dropout levels, weight decay values, and batch sizes.
Edit or extend this cell freely — the sweep will pick the best automatically.

In [ ]:
HYPERPARAM_GRID = [
    # id  description
    # 0   baseline
    {'lr': 3e-4, 'dropout': 0.10, 'weight_decay': 1e-3, 'batch_size': 16, 'num_epochs': 20},
    # 1   lower lr — finer updates
    {'lr': 1e-4, 'dropout': 0.20, 'weight_decay': 1e-4, 'batch_size': 16, 'num_epochs': 20},
    # 2   higher lr + larger batch
    {'lr': 5e-4, 'dropout': 0.30, 'weight_decay': 1e-3, 'batch_size': 32, 'num_epochs': 20},
    # 3   small batch — noisy gradients act as regularizer
    {'lr': 3e-4, 'dropout': 0.20, 'weight_decay': 1e-2, 'batch_size':  8, 'num_epochs': 20},
    # 4   aggressive lr — fast convergence
    {'lr': 1e-3, 'dropout': 0.10, 'weight_decay': 1e-4, 'batch_size': 16, 'num_epochs': 15},
    # 5   sweet-spot: moderate lr + light decay
    {'lr': 2e-4, 'dropout': 0.15, 'weight_decay': 5e-4, 'batch_size': 16, 'num_epochs': 20},
    # 6   large batch + low dropout — less noise
    {'lr': 5e-4, 'dropout': 0.10, 'weight_decay': 1e-3, 'batch_size': 32, 'num_epochs': 20},
    # 7   conservative — very low lr, longer run
    {'lr': 5e-5, 'dropout': 0.20, 'weight_decay': 1e-4, 'batch_size': 16, 'num_epochs': 25},
]

# Preview the grid
df_grid = pd.DataFrame(HYPERPARAM_GRID)
df_grid.index.name = 'id'
display(df_grid)

## 11 · Training loop
Each config runs with **early stopping** (patience = 4 epochs). Best checkpoint per run is kept.

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

EARLY_STOP_PATIENCE = 4   # ← tweak if desired


def train_config(config, train_samples, val_samples, run_id, results_dir=RESULTS_DIR):
    print(f"\n{'='*62}")
    print(f'  Run {run_id} | {config}')
    print(f"{'='*62}")

    train_loader = DataLoader(
        CustomDataset(train_samples, get_transforms(True)),
        batch_size=config['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader = DataLoader(
        CustomDataset(val_samples, get_transforms(False)),
        batch_size=config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    model = CNNViTHybrid(dropout=config['dropout']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['num_epochs'])
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, 1.2, 1.0, 2.5]).to(DEVICE), label_smoothing=0.1
    )

    weights_path = os.path.join(results_dir, f'model_run{run_id}.pth')
    best_val_acc = 0.0
    no_improve   = 0
    history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, config['num_epochs'] + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        v_loss, v_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        flag = ''
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            no_improve   = 0
            torch.save(model.state_dict(), weights_path)
            flag = '  ← best'
        else:
            no_improve += 1

        print(f'  Epoch {epoch:03d}/{config["num_epochs"]} | '
              f'Train {t_loss:.4f}/{t_acc:.4f}  Val {v_loss:.4f}/{v_acc:.4f}{flag}')

        if no_improve >= EARLY_STOP_PATIENCE:
            print(f'  [Early stop] No improvement for {EARLY_STOP_PATIENCE} epochs.')
            break

    # Per-class eval on best checkpoint
    model.load_state_dict(torch.load(weights_path, weights_only=True))
    preds, lbls = evaluate_per_class(model, val_loader, DEVICE)
    per_class_acc = compute_per_class_accuracy(preds, lbls)
    plot_confusion_heatmap(preds, lbls,
                           save_path=os.path.join(results_dir, f'heatmap_run{run_id}.png'))

    epochs_run = len(history['val_acc'])
    return {
        'run_id':        run_id,
        'config':        config,
        'best_val_acc':  best_val_acc,
        'per_class_acc': per_class_acc,
        'epochs_run':    epochs_run,
        'history':       history,
    }

In [ ]:
all_results = []
for i, cfg in enumerate(HYPERPARAM_GRID):
    result = train_config(cfg, train_samples, val_samples, run_id=i)
    all_results.append(result)
print(f'\nAll {len(all_results)} configs done.')

## 12 · Select best config
Ranked leaderboard — all configs sorted by validation accuracy. Best weights are promoted to `model_weights.pth`.

In [ ]:
ranked = sorted(all_results, key=lambda r: r['best_val_acc'], reverse=True)

rows = []
for rank, r in enumerate(ranked, 1):
    row = {
        'Rank':         rank,
        'Run':          f"run{r['run_id']}",
        'Val Acc':      r['best_val_acc'],
        'Epochs Run':   r['epochs_run'],
        'LR':           r['config']['lr'],
        'Dropout':      r['config']['dropout'],
        'Weight Decay': r['config']['weight_decay'],
        'Batch Size':   r['config']['batch_size'],
    }
    row.update({f'acc_{k}': v for k, v in r['per_class_acc'].items()})
    rows.append(row)

df_lb = pd.DataFrame(rows).set_index('Rank')
display(
    df_lb.style
        .background_gradient(cmap='YlGn',   subset=['Val Acc'])
        .background_gradient(cmap='Blues',  subset=[c for c in df_lb.columns if c.startswith('acc_')])
        .format({'Val Acc': '{:.4f}', **{c: '{:.4f}' for c in df_lb.columns if c.startswith('acc_')}})
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

palette = sns.color_palette('tab10', len(all_results))
for r, c in zip(ranked, palette):
    lbl = f"run{r['run_id']} acc={r['best_val_acc']:.3f}"
    lw  = 2.5 if r is ranked[0] else 1.0
    axes[0].plot(r['history']['val_acc'], label=lbl, color=c, linewidth=lw)
axes[0].set_title('Validation Accuracy per Epoch'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy'); axes[0].legend(fontsize=7)

run_labels = [f"run{r['run_id']}" for r in ranked]
accs       = [r['best_val_acc'] for r in ranked]
colors     = [palette[i] for i in range(len(ranked))]
bars = axes[1].bar(run_labels, accs, color=colors)
axes[1].set_title('Best Val Accuracy — ranked'); axes[1].set_ylabel('Accuracy')
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{acc:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'run_comparison.png'), dpi=150)
plt.show()

In [ ]:
best = ranked[0]   # highest val_acc after sorting

print('=' * 62)
print(f'  BEST CONFIG  →  run{best["run_id"]}  |  val_acc = {best["best_val_acc"]:.4f}')
print('=' * 62)
print(f'  LR           : {best["config"]["lr"]}')
print(f'  Dropout      : {best["config"]["dropout"]}')
print(f'  Weight Decay : {best["config"]["weight_decay"]}')
print(f'  Batch Size   : {best["config"]["batch_size"]}')
print(f'  Epochs Run   : {best["epochs_run"]} / {best["config"]["num_epochs"]}')
print(f'  Per-class    : {best["per_class_acc"]}')
print('=' * 62)

# Promote best weights to model_weights.pth
best_src = os.path.join(RESULTS_DIR, f"model_run{best['run_id']}.pth")
best_dst = os.path.join(RESULTS_DIR, 'model_weights.pth')
shutil.copy2(best_src, best_dst)
print(f'\nBest weights → {best_dst}')

# Save best config as its own JSON for quick reference
with open(os.path.join(RESULTS_DIR, 'best_config.json'), 'w') as f:
    json.dump({'run_id': best['run_id'], 'config': best['config'],
               'best_val_acc': best['best_val_acc'],
               'per_class_acc': best['per_class_acc']}, f, indent=2)

# Full sweep summary
summary = [{'run_id': r['run_id'], 'config': r['config'],
            'best_val_acc': r['best_val_acc'], 'epochs_run': r['epochs_run'],
            'per_class_acc': r['per_class_acc']} for r in all_results]
with open(os.path.join(RESULTS_DIR, 'hyperparam_results.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Sweep summary → {os.path.join(RESULTS_DIR, "hyperparam_results.json")}')

## 13 · Save to Google Drive
Creates `iter<N>_<hash>/` — best weights, all per-run weights, heatmaps, comparison plot, and both JSON files.

In [ ]:
saved_to = save_to_gdrive(RESULTS_DIR)
print(f'\nDrive folder: {saved_to}')

## 14 · Inference on a single image

In [ ]:
@torch.no_grad()
def predict(model, image_path, device=DEVICE):
    img = get_transforms(False)(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    model.eval()
    return ID_TO_LABEL[model(img).argmax(1).item()]


# Example — load best model and run inference
# best_model = CNNViTHybrid().to(DEVICE)
# best_model.load_state_dict(torch.load(best_dst, weights_only=True))
# print(predict(best_model, '/path/to/image.jpg'))